

## Explanation of A2C

A2C has two parts.

### Actor

The **actor** chooses the action.

Example actions in MountainCar:

```text
0 = push left
1 = no push
2 = push right
```

The actor gives probabilities for these actions.

### Critic

The **critic** checks how good the current state is.

It gives a value number. This value tells us whether the car is in a good or bad situation.

### Advantage

The **advantage** tells us:

> Was the selected action better or worse than expected?

If the action was better than expected, the actor should do it more often.

If the action was worse than expected, the actor should do it less often.

# Step 1: Install required libraries

Run this cell only if Gymnasium is not already installed.

In [12]:
# Uncomment and run this cell if needed
!pip install gymnasium[classic-control] torch matplotlib numpy

# Step 2: Import libraries

- `gymnasium` creates the environment.
- `torch` builds and trains the neural network.
- `numpy` helps with calculations.
- `matplotlib` draws graphs.

In [13]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

import random

# Step 3: Set random seed

A random seed helps us get similar results when we run the notebook again.

Reinforcement Learning is still random, so results may not be exactly the same every time.

In [14]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Seed set successfully")

Seed set successfully


# Step 4: Create the MountainCar environment

The observation has two numbers:

1. **Position** of the car.
2. **Velocity** of the car.

The action has three choices:

```text
0 = push left
1 = no push
2 = push right
```

In [15]:
env = gym.make("MountainCar-v0")
state, info = env.reset(seed=SEED)
print("Initial state:", state)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Number of actions:", env.action_space.n)
env.close()

Initial state: [-0.4452088  0.       ]
Observation space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Action space: Discrete(3)
Number of actions: 3


# Step 5: Why MountainCar is difficult

In MountainCar, the original reward is:

```text
-1 at every step
```

If the car fails for 200 steps, total reward is:

```text
-200
```

If the car reaches the flag earlier, the reward is better, for example:

```text
-150
```

This is difficult because the agent does not get much guidance while learning.

So we use **reward shaping** during training.

Reward shaping means:

> We add helpful reward signals to guide the agent.

But during testing, we still check the real result:

> Did the car reach the flag?

# Step 6: Reward shaping function

The car must build momentum.

So we give extra reward when:

1. The car gains speed.
2. The car moves closer to the goal.
3. The car reaches the flag.

In [16]:
def shaped_reward(state, next_state, reward, terminated):
    # next_state contains position and velocity
    position = next_state[0]
    velocity = next_state[1]

    # Start with the original reward
    new_reward = reward

    # Encourage speed, because the car needs momentum
    new_reward += 10.0 * abs(velocity)

    # Encourage movement toward the right side
    new_reward += 2.0 * (position + 0.5)

    # Give a large bonus if the flag is reached
    if terminated:
        new_reward += 100.0

    return new_reward


# Step 7: Build the Actor-Critic neural network

A2C uses one network with two outputs.

### Actor output

The actor gives action scores. We convert these scores into probabilities using softmax.

### Critic output

The critic gives one value number. This number estimates how good the current state is.

So the structure is:

```text
Shared layers → Actor head → action probabilities
Shared layers → Critic head → state value
```

In [17]:
class ActorCriticNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )

        self.actor = nn.Linear(128, action_size)
        self.critic = nn.Linear(128, 1)

    def forward(self, state):
        x = self.shared(state)
        action_logits = self.actor(x)
        state_value = self.critic(x)
        return action_logits, state_value

# Step 8: Test the network once

Before training, we check that the network output is correct.

For one state:

- Actor should give 3 action scores.
- Critic should give 1 value.

In [18]:
env = gym.make("MountainCar-v0")
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

a2c_model = ActorCriticNetwork(state_size, action_size)
state, info = env.reset(seed=SEED)
state_tensor = torch.FloatTensor(state).unsqueeze(0)

logits, value = a2c_model(state_tensor)
probabilities = torch.softmax(logits, dim=-1)

print("State size:", state_size)
print("Action size:", action_size)
print("Action logits:", logits.detach().numpy())
print("Action probabilities:", probabilities.detach().numpy())
print("State value:", value.item())
env.close()

State size: 2
Action size: 3
Action logits: [[-0.01644429  0.19481304  0.00635532]]
Action probabilities: [[0.3069093  0.37910363 0.31398708]]
State value: 0.03332221880555153


# Step 9: A2C learning idea

For each step, the agent does this:

1. Observe the current state.
2. Actor chooses an action.
3. Environment gives next state and reward.
4. Critic estimates the value of current state and next state.
5. Calculate advantage.
6. Update actor and critic.

The advantage is:

```text
advantage = target value - current value
```

For one-step A2C, the target value is:

```text
reward + gamma × next_state_value
```

If advantage is positive, the action was better than expected.

If advantage is negative, the action was worse than expected.

# Step 10: Training function for A2C

This training function includes:

- reward shaping,
- entropy bonus for exploration,
- gradient clipping for stability,
- success counting,
- saving the best model,
- early stopping when the agent reaches the flag often.

Entropy encourages exploration. It stops the agent from always repeating the same poor action too early.

In [19]:
def train_a2c(
    max_episodes=4000,
    gamma=0.99,
    learning_rate=0.0005,
    entropy_beta=0.01,
    target_success_rate=0.60,
    check_window=100,
    model_path="best_a2c_mountaincar.pth"
):
    env = gym.make("MountainCar-v0")

    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n

    model = ActorCriticNetwork(state_size, action_size)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    original_rewards = []
    shaped_rewards_list = []
    success_history = []
    best_success_rate = -1
    best_average_reward = -9999

    for episode in range(max_episodes):
        state, info = env.reset(seed=SEED + episode)
        done = False
        total_original_reward = 0
        total_shaped_reward = 0
        reached_flag = False

        while not done:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)

            logits, value = model(state_tensor)
            probabilities = torch.softmax(logits, dim=-1)
            distribution = Categorical(probabilities)

            action = distribution.sample()
            log_probability = distribution.log_prob(action)
            entropy = distribution.entropy()

            next_state, reward, terminated, truncated, info = env.step(action.item())
            done = terminated or truncated

            if terminated:
                reached_flag = True

            train_reward = shaped_reward(state, next_state, reward, terminated)

            next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0)
            with torch.no_grad():
                _, next_value = model(next_state_tensor)

            if terminated:
                target_value = torch.tensor([[train_reward]], dtype=torch.float32)
            else:
                target_value = torch.tensor([[train_reward]], dtype=torch.float32) + gamma * next_value

            advantage = target_value - value

            actor_loss = -log_probability * advantage.detach().squeeze()
            critic_loss = advantage.pow(2).mean()
            entropy_loss = -entropy_beta * entropy.mean()
            loss = actor_loss + 0.5 * critic_loss + entropy_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            state = next_state
            total_original_reward += reward
            total_shaped_reward += train_reward

        original_rewards.append(total_original_reward)
        shaped_rewards_list.append(total_shaped_reward)
        success_history.append(1 if reached_flag else 0)

        if len(original_rewards) >= check_window:
            avg_original_reward = np.mean(original_rewards[-check_window:])
            success_rate = np.mean(success_history[-check_window:])

            if success_rate > best_success_rate or avg_original_reward > best_average_reward:
                best_success_rate = success_rate
                best_average_reward = avg_original_reward
                torch.save(model.state_dict(), model_path)

            if (episode + 1) % 50 == 0:
                print(
                    f"Episode {episode+1:4d} | "
                    f"Avg original reward: {avg_original_reward:7.2f} | "
                    f"Success rate: {success_rate*100:5.1f}%"
                )

            if success_rate >= target_success_rate:
                print("Training stopped early because the agent is reaching the flag often.")
                print(f"Success rate over last {check_window} episodes: {success_rate*100:.1f}%")
                torch.save(model.state_dict(), model_path)
                break
        else:
            if (episode + 1) % 50 == 0:
                print(f"Episode {episode+1:4d} | Collecting first {check_window} episodes...")

    env.close()
    print("A2C training completed")
    print("Best model saved as:", model_path)
    return model, original_rewards, shaped_rewards_list, success_history

# Step 11: Train the A2C model

A2C usually learns faster than REINFORCE because the critic gives feedback at every step.

For classroom demonstration, start with 4000 episodes.

If the model still fails, increase `max_episodes` to 6000 or 8000.

In [ ]:
a2c_model, a2c_original_rewards, a2c_shaped_rewards, a2c_success_history = train_a2c(
    max_episodes=4000,
    gamma=0.99,
    learning_rate=0.0005,
    entropy_beta=0.01,
    target_success_rate=0.60,
    check_window=100,
    model_path="best_a2c_mountaincar.pth"
)

Episode   50 | Collecting first 100 episodes...
Episode  100 | Avg original reward: -200.00 | Success rate:   0.0%


# Step 12: Plot original reward

This plot shows the original MountainCar reward.

- `-200` means failure.
- Better than `-200` means improvement.
- Reaching the goal earlier gives a higher reward.

In [ ]:
def moving_average(values, window=50):
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 5))
plt.plot(a2c_original_rewards, label="Original reward per episode", alpha=0.4)
plt.plot(moving_average(a2c_original_rewards, 50), label="Moving average over 50 episodes")
plt.xlabel("Episode")
plt.ylabel("Original reward")
plt.title("A2C Training on MountainCar-v0")
plt.legend()
plt.grid(True)
plt.show()

# Step 13: Plot success rate

A value near 1 means the car reaches the flag often.

A value near 0 means the car usually fails.

In [ ]:
success_rate_curve = moving_average(a2c_success_history, 100)

plt.figure(figsize=(10, 5))
plt.plot(success_rate_curve, label="Success rate over 100 episodes")
plt.xlabel("Episode")
plt.ylabel("Success rate")
plt.title("A2C Success Rate")
plt.legend()
plt.grid(True)
plt.show()

# Step 14: Load the best saved model

During training, we saved the best model.

Now we load it before testing.

In [ ]:
def load_a2c_model(model_path="best_a2c_mountaincar.pth"):
    env = gym.make("MountainCar-v0")
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n
    env.close()

    model = ActorCriticNetwork(state_size, action_size)
    model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))
    model.eval()
    return model

best_a2c_model = load_a2c_model("best_a2c_mountaincar.pth")
print("Best A2C model loaded successfully")

# Step 15: Test the trained model

Testing is different from training.

During testing:

- We do not update the model.
- We do not use reward shaping.
- We check if the car really reaches the flag.

For testing, we choose the action with the highest probability. This is called **greedy action selection**.

In [ ]:
def test_a2c_model(model, test_episodes=20, render=False):
    if render:
        env = gym.make("MountainCar-v0", render_mode="human")
    else:
        env = gym.make("MountainCar-v0")

    rewards = []
    successes = []
    steps_list = []

    for episode in range(test_episodes):
        state, info = env.reset(seed=1000 + episode)
        done = False
        total_reward = 0
        steps = 0
        reached_flag = False

        while not done:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)

            with torch.no_grad():
                logits, value = model(state_tensor)
                probabilities = torch.softmax(logits, dim=-1)
                action = torch.argmax(probabilities, dim=-1).item()

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            if terminated:
                reached_flag = True

            state = next_state
            total_reward += reward
            steps += 1

        rewards.append(total_reward)
        successes.append(1 if reached_flag else 0)
        steps_list.append(steps)

        print(
            f"Test episode {episode+1:2d} | "
            f"Reward: {total_reward:6.1f} | "
            f"Steps: {steps:3d} | "
            f"Reached flag: {reached_flag}"
        )

    env.close()
    print("Testing summary")
    print("Average reward:", np.mean(rewards))
    print("Success rate:", np.mean(successes) * 100, "%")
    print("Average steps:", np.mean(steps_list))
    return rewards, successes, steps_list

# Step 16: Run model testing

This cell tests the model for 20 episodes.

If success rate is low, train for more episodes.

In [ ]:
test_rewards, test_successes, test_steps = test_a2c_model(best_a2c_model, test_episodes=20, render=False)

# Step 17: Visual test

This cell opens a window and shows the car moving.

Use this only on a local Jupyter notebook.

It may not work properly in Google Colab.

In [ ]:
# Run this only if your system supports rendering windows.
# test_a2c_model(best_a2c_model, test_episodes=3, render=True)

# Step 18: What to do if the car still fails

A2C is better than REINFORCE for this problem, but training can still be unstable.

Try these changes one by one:

## Option 1: Increase training episodes

```python
max_episodes=6000
```

or

```python
max_episodes=8000
```

## Option 2: Change learning rate

```python
learning_rate=0.0003
```

or

```python
learning_rate=0.001
```

## Option 3: Increase exploration

```python
entropy_beta=0.02
```

## Option 4: Lower early stopping target for classroom demo

```python
target_success_rate=0.40
```

---

# Teacher explanation for students

> A2C uses a critic, so it gets feedback more often than REINFORCE. The actor learns which action to choose, while the critic learns how good the current state is. Because of this, A2C usually learns faster and more stably than REINFORCE.

# Step 19: Student questions

Ask students:

1. What is the job of the actor?
2. What is the job of the critic?
3. Why do we use softmax in the actor?
4. What does advantage mean?
5. Why is MountainCar difficult?
6. Why did we use reward shaping?
7. How is A2C different from REINFORCE?

---

# Step 20: Final comparison with REINFORCE

| Feature | REINFORCE | A2C |
|---|---|---|
| Main idea | Learns from full episode return | Learns step by step using critic |
| Feedback | After episode | After each step |
| Stability | Less stable | More stable |
| Speed | Usually slower | Usually faster |
| Uses critic? | No | Yes |
| Good for MountainCar? | Difficult | Better than REINFORCE |

---

# Lab conclusion

In this lab, we trained and tested an A2C agent on MountainCar.

The most important idea is:

> The actor chooses actions, and the critic helps the actor learn whether those actions were good or bad.